In [1]:
import os
import pandas as pd
from tqdm import tqdm


class AISDataLoader:
    """
    Handles efficient loading of multiple AIS CSV files into pandas DataFrames.
    Includes memory-safe chunk loading support.
    """

    def __init__(self, base_folder: str):
        self.base_folder = base_folder
        self.ais_dfs = {}

    def _generate_file_paths(self, days=range(3, 8)):
        """Generate AIS file paths for given days (default 03–07)."""
        csv_files = [
            os.path.join(
                self.base_folder, f"AIS_2020_01_{day:02d}", f"AIS_2020_01_{day:02d}.csv"
            )
            for day in days
        ]
        return csv_files

    def load_all(self, days=range(1, 9), use_chunks=False, chunk_size=500_000):
        """
        Load multiple AIS CSVs.
        - If `use_chunks=True`, reads in smaller pieces and concatenates.
        """
        csv_files = self._generate_file_paths(days)

        for file_path in tqdm(csv_files, desc="📂 Loading AIS CSVs"):
            name = os.path.basename(file_path).replace(".csv", "")

            try:
                if use_chunks:
                    chunks = []
                    for chunk in pd.read_csv(
                        file_path, chunksize=chunk_size, low_memory=False
                    ):
                        chunks.append(chunk)
                    df = pd.concat(chunks, ignore_index=True)
                    del chunks
                else:
                    df = pd.read_csv(file_path, low_memory=False)

                self.ais_dfs[name] = df
                print(f"✅ Loaded {name}: {len(df):,} rows")

            except Exception as e:
                print(f"❌ Error loading {file_path}: {e}")

        return self.ais_dfs


class AISProcessor:
    """
    Handles cleaning and filtering AIS DataFrames.
    """

    def __init__(self, ais_dfs: dict):
        self.ais_dfs = ais_dfs

    @staticmethod
    def convert_datetime(df: pd.DataFrame) -> pd.DataFrame:
        """Convert BaseDateTime to pandas datetime safely."""
        df["BaseDateTime"] = pd.to_datetime(
            df["BaseDateTime"], format="%Y-%m-%dT%H:%M:%S", errors="coerce"
        )
        return df

    def filter_by_points(self, min_points: int = 50):
        """
        Remove MMSIs with fewer than `min_points` AIS records.
        """
        updated = {}

        for name, df in tqdm(self.ais_dfs.items(), desc="🔍 Filtering datasets"):
            print(f"\nProcessing {name}...")
            original_size = len(df)

            # Convert datetime safely
            df = self.convert_datetime(df)

            # Count points per MMSI
            mmsi_counts = df["MMSI"].value_counts()
            valid_mmsis = mmsi_counts[mmsi_counts >= min_points].index

            # Filter valid MMSIs
            df_filtered = df[df["MMSI"].isin(valid_mmsis)].reset_index(drop=True)

            print(
                f"  Original: {original_size:,} → Filtered: {len(df_filtered):,} rows "
                f"(Removed {original_size - len(df_filtered):,})"
            )

            updated[name] = df_filtered

            # Free memory
            del df, df_filtered, mmsi_counts, valid_mmsis

        self.ais_dfs = updated
        return self.ais_dfs


class AISPipeline:
    """
    High-level orchestration class for the AIS workflow.
    """

    def __init__(
        self, base_folder: str, min_points: int = 50, use_chunks: bool = False
    ):
        self.base_folder = base_folder
        self.min_points = min_points
        self.use_chunks = use_chunks
        self.loader = AISDataLoader(base_folder)
        self.processor = None
        self.ais_dfs = {}

    def run(self):
        """Run the complete pipeline."""
        print("\n🚀 Starting AIS data processing pipeline...\n")

        # Step 1: Load
        ais_dfs = self.loader.load_all(use_chunks=self.use_chunks)
        print("\n✅ All CSVs loaded.\n")

        # Step 2: Process
        self.processor = AISProcessor(ais_dfs)
        filtered_dfs = self.processor.filter_by_points(self.min_points)

        # Store processed data
        self.ais_dfs = filtered_dfs
        print("\n🏁 Pipeline complete.\n")
        return self.ais_dfs


# Example usage:
if __name__ == "__main__":
    data_folder = r"D:\Maritime_Vessel_monitoring\csv_extracted_data"

    pipeline = AISPipeline(
        base_folder=data_folder,
        min_points=50,
        use_chunks=True,  # ✅ safer for large files
    )

    ais_data = pipeline.run()


🚀 Starting AIS data processing pipeline...



📂 Loading AIS CSVs:  12%|█▎        | 1/8 [00:01<00:11,  1.67s/it]

✅ Loaded AIS_2020_01_01: 1,048,575 rows


📂 Loading AIS CSVs:  25%|██▌       | 2/8 [00:03<00:10,  1.71s/it]

✅ Loaded AIS_2020_01_02: 1,048,575 rows


...

  Original: 7,032,498 → Filtered: 7,003,855 rows (Removed 28,643)

Processing AIS_2020_01_07...


🔍 Filtering datasets:  88%|████████▊ | 7/8 [00:14<00:02,  2.50s/it]

  Original: 6,808,529 → Filtered: 6,782,070 rows (Removed 26,459)

Processing AIS_2020_01_08...


🔍 Filtering datasets: 100%|██████████| 8/8 [00:17<00:00,  2.19s/it]

  Original: 6,870,094 → Filtered: 6,844,953 rows (Removed 25,141)

🏁 Pipeline complete.



In [3]:
df_1 = ais_data["AIS_2020_01_01"]
# Example
df_1

,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass
0,538008468,2020-01-01 00:00:00,38.25802,-76.29487,14.9,338.6,337,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,A
1,368120510,2020-01-01 00:00:00,27.58610,-82.75991,0.0,58.6,511,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,A
2,368063930,2020-01-01 00:00:00,40.71045,-73.97588,11.2,208.9,207,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,A
3,368106220,2020-01-01 00:00:00,38.53932,-90.25523,0.2,161.8,31,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,NaN,A
4,367336180,2020-01-01 00:00:00,56.02945,-132.68705,9.0,325.5,511,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,NaN,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
988444,368108120,2020-01-01 04:13:51,35.25534,-90.09801,4.6,339.4,340,PORTER J. FURLONG,NaN,WDK9974,31.0,0.0,264.0,16.0,3.0,31.0,A
988445,367447530,2020-01-01 04:16:21,27.70220,-82.71770,0.1,154.3,511,SWEET MELISSA,NaN,WDF4063,37.0,NaN,14.0,4.0,NaN,NaN,B
988446,367379540,2020-01-01 04:17:15,29.83253,-91.17663,3.6,325.9,511,EMMANUEL,NaN,WDE6384,31.0,0.0,21.0,9.0,NaN,57.0,A
988447,366963050,2020-01-01 04:16:25,38.92215,-90.28195,4.7,271.8,511,DALE A HELLER,NaN,WDB8703,31.0,0.0,39.0,13.0,NaN,NaN,A


In [ ]:
ais_data = {
    "AIS_2020_01_03": ais_data["AIS_2020_01_03"],
    "AIS_2020_01_04": ais_data["AIS_2020_01_04"],
    "AIS_2020_01_05": ais_data["AIS_2020_01_05"],
    "AIS_2020_01_06": ais_data["AIS_2020_01_06"],
    "AIS_2020_01_07": ais_data["AIS_2020_01_07"],
    "AIS_2020_01_08": ais_data["AIS_2020_01_08"],
    # ... and so on
}

In [ ]:
import os
import pandas as pd
import torch
import warnings
from tqdm import tqdm
import time

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True  # cuDNN autotuner for faster GPU ops


# ----------------------------------
# Vessel feature engineering class
# ----------------------------------
class VesselFeatureEngineer:
    def __init__(self, sog_stop_threshold=0.5, stop_duration_min=5, device=None):
        self.sog_stop_threshold = sog_stop_threshold
        self.stop_duration_min = stop_duration_min
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    def process(self, df: pd.DataFrame) -> pd.DataFrame:
        if len(df) == 0:
            return df
        df = df.copy().sort_values("BaseDateTime")

        # Timestamp features
        df["hour"] = df["BaseDateTime"].dt.hour
        df["dayofweek"] = df["BaseDateTime"].dt.dayofweek
        df["month"] = df["BaseDateTime"].dt.month
        df["is_weekend"] = df["dayofweek"] >= 5

        # Numeric tensor
        numeric_cols = ["SOG", "COG", "Heading"]
        tensor = torch.tensor(
            df[numeric_cols].fillna(0).values, dtype=torch.float32, device=self.device
        )

        # Velocity components
        cog_rad = tensor[:, 1] * (torch.pi / 180)
        df["v_x"] = (tensor[:, 0] * torch.cos(cog_rad)).cpu().numpy()
        df["v_y"] = (tensor[:, 0] * torch.sin(cog_rad)).cpu().numpy()

        # Time deltas in hours
        timestamps = torch.tensor(
            df["BaseDateTime"].astype("int64").values // 1e9,
            dtype=torch.float32,
            device=self.device,
        )
        delta_t = torch.zeros_like(timestamps)
        delta_t[1:] = (timestamps[1:] - timestamps[:-1]) / 3600.0

        # Turn rate
        delta_cog = torch.zeros_like(tensor[:, 1])
        delta_cog[1:] = tensor[1:, 1] - tensor[:-1, 1]
        delta_cog[delta_cog > 180] -= 360
        delta_cog[delta_cog < -180] += 360
        df["turn_rate"] = (delta_cog / delta_t).cpu().numpy()

        # Acceleration
        delta_sog = torch.zeros_like(tensor[:, 0])
        delta_sog[1:] = tensor[1:, 0] - tensor[:-1, 0]
        df["accel_knots_per_hr"] = (delta_sog / delta_t).cpu().numpy()

        # Stop detection
        is_slow = tensor[:, 0] < self.sog_stop_threshold
        changes = torch.diff(is_slow.to(torch.int))
        stop_group = torch.zeros_like(is_slow, dtype=torch.long)
        stop_group[1:] = torch.cumsum(changes != 0, dim=0)

        unique_groups = torch.unique(stop_group)
        group_start_idx = torch.zeros_like(unique_groups)
        group_end_idx = torch.zeros_like(unique_groups)
        for i, g in enumerate(unique_groups):
            idxs = (stop_group == g).nonzero(as_tuple=True)[0]
            group_start_idx[i] = idxs[0]
            group_end_idx[i] = idxs[-1]

        durations_min = (timestamps[group_end_idx] - timestamps[group_start_idx]) / 60.0
        group_slow = is_slow[group_start_idx]
        stopped_groups = unique_groups[
            (durations_min >= self.stop_duration_min) & group_slow
        ]
        df["is_stopped"] = torch.isin(stop_group, stopped_groups).cpu().numpy()

        # Extra features
        df["Δt_hours"] = delta_t.cpu().numpy()
        df["ΔCOG"] = delta_cog.cpu().numpy()
        df["ΔSOG"] = delta_sog.cpu().numpy()
        df["COG_rad"] = cog_rad.cpu().numpy()

        return df


# ----------------------------------
# AIS pipeline: process daywise & save
# ----------------------------------
class AISFeaturePipeline:
    def __init__(
        self, ais_data: dict, output_dir="processed_data", device=None, **kwargs
    ):
        self.ais_data = ais_data
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.engineer = VesselFeatureEngineer(device=self.device, **kwargs)

    def process_and_store_day(self, day_name, df):
        """Process one day and store efficiently."""
        enriched_vessels = []

        for mmsi, vessel_df in df.groupby("MMSI"):
            try:
                enriched = self.engineer.process(vessel_df)
                enriched_vessels.append(enriched)
            except Exception as e:
                print(f"Error processing MMSI {mmsi}: {e}")
                continue

        if not enriched_vessels:
            return None

        day_df = pd.concat(enriched_vessels, ignore_index=True)

        # Optional: one-hot encoding for small cardinality columns only
        for col in ["Vessel Name", "IMO", "CallSignTranscription"]:
            if col in day_df.columns:
                dummies = pd.get_dummies(day_df[col], prefix=col, sparse=True)
                day_df = pd.concat([day_df, dummies], axis=1)
                day_df.drop(columns=[col], inplace=True)

        # Save as pickle (memory-efficient)
        path = os.path.join(self.output_dir, f"{day_name}.pkl")
        day_df.to_pickle(path)
        return path

    def process_all(self):
        paths = {}
        start_time = time.time()
        for day_name, df in tqdm(
            self.ais_data.items(), desc="Processing days", total=len(self.ais_data)
        ):
            tqdm.write(f"Processing {day_name} ({len(df):,} rows)")
            path = self.process_and_store_day(day_name, df)
            if path:
                paths[day_name] = path

            # Clear GPU memory
            torch.cuda.empty_cache()

        total_time = time.time() - start_time
        print(f"\n✅ All processing done in {total_time:.2f} seconds")
        return paths


# ----------------------------------
# Run pipeline
# ----------------------------------
pipeline = AISFeaturePipeline(ais_data=ais_data)
daywise_paths = pipeline.process_all()

# Example: load back a specific day
# day_df = pd.read_pickle(daywise_paths["2025-10-01"])

Processing days:   0%|          | 0/6 [00:00<?, ?it/s]

Processing AIS_2020_01_03 (7,089,446 rows)


Processing days:  17%|█▋        | 1/6 [02:41<13:27, 161.59s/it]

Processing AIS_2020_01_04 (6,908,736 rows)


Processing days:  33%|███▎      | 2/6 [05:25<10:51, 162.77s/it]

...

Processing AIS_2020_01_07 (6,782,070 rows)


Processing days:  83%|████████▎ | 5/6 [14:04<02:52, 172.29s/it]

Processing AIS_2020_01_08 (6,844,953 rows)


Processing days: 100%|██████████| 6/6 [16:50<00:00, 168.44s/it]


✅ All processing done in 1010.66 seconds
